In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("../extraction/assignmentsRaw.csv", sep=";")

df

In [ ]:
def eda_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "count": df.count(),
        "missing": df.isna().sum(),
        "missing_%": df.isna().mean() * 100,
        "unique": df.nunique(dropna=True),
    })

    summary["min"] = df.select_dtypes(include="number").min()
    summary["mean"] = df.select_dtypes(include="number").mean()
    summary["median"] = df.select_dtypes(include="number").median()
    summary["max"] = df.select_dtypes(include="number").max()

    return summary.round(2)

summary = eda_summary(df)
summary

In [ ]:
df_dropped = df.drop(columns= summary[summary["missing"]>0].index)

In [ ]:

df_dropped["startDate"] = pd.to_datetime(
    df_dropped["startDate"].astype("string"),
    format="%Y%m%d%H%M%S",
    errors="coerce"
)

df_dropped["endDate"] = pd.to_datetime(
    df_dropped["endDate"].astype("string"),
    format="%Y%m%d%H%M%S",
    errors="coerce"
)


In [ ]:
summary = eda_summary(df_dropped)

dtypes = summary["dtype"].astype(str).value_counts()

plt.bar(
    dtypes.index,
    dtypes.values,
    edgecolor="black"
)

plt.title("Distribution of Data Types in Features")
plt.xlabel("Data Type")
plt.ylabel("Number of Columns")

plt.yticks(
    np.arange(0, 6, 1)
)

plt.tight_layout()
plt.show()

In [ ]:
feature_summary = pd.DataFrame({
    "nunique": df_dropped.nunique(),
    "most_common_pct": df_dropped.apply(
        lambda col: col.value_counts(normalize=True, dropna=False).iloc[0] * 100
    )
})

print(feature_summary.sort_values("most_common_pct", ascending=False))

plt.hist(
    feature_summary["most_common_pct"],
    bins=40,
    edgecolor="black"
)

plt.title("Distribution of Most Common Value Percentage Across Features")
plt.xlabel("Most Common Value (%)")
plt.ylabel("Number of Features")

plt.xticks(np.arange(0, 101, 20))

# Add a little visual space after 100
plt.xlim(-3, 103)

plt.show()

In [ ]:
corr = df_dropped.drop(columns=["id", "assignee", "clients"]).corr()

# Hide upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Dynamic figure size
n_features = len(corr.columns)
figsize = max(12, n_features * 0.45)

fig, ax = plt.subplots(figsize=(figsize, figsize))


im = ax.imshow(
    corr,
    vmin=-1,
    vmax=1,
    aspect="equal"
)

fig.colorbar(im, ax=ax, label="Correlation")

ax.set_xticks(range(n_features))
ax.set_yticks(range(n_features))

ax.set_xticklabels(
    corr.columns,
    rotation=90,
    fontsize=15
)

ax.set_yticklabels(
    corr.columns,
    fontsize=15
)

ax.set_title(
    "Correlation Matrix of Model Features",
    fontsize=18,
    pad=20
)

plt.tight_layout()
plt.show()

In [ ]:
colsToCombine = [
    "arrivalSelfOrganized",
	"departureSelfOrganized",
	"arrivalNoBilling",
	"departureNoBilling"
]

colsToCombine


df_dropped["selfOrganized"] = (
    df_dropped[colsToCombine]
    .mean(axis=1)
    .round()
)


df_toExport = df_dropped.drop(columns=colsToCombine)

In [ ]:
df_toExport.info()

In [ ]:
df_toExport.to_csv("assignments.csv", sep=";", index=False)